# AAT6153 Chapter 2 — Hands-on 5
## Character-Level Text Generation with LSTM
**Slides:** Hands-on 5 (75–78)

Working with **characters** keeps the vocabulary tiny (65 symbols for this text), so there are no unknown words, and the model must learn spelling, punctuation and layout by itself (Karpathy's char-rnn, 2015).

In this notebook you will:
1. Map each character of Shakespeare's works to an integer (slide 75).
2. Cut the text into **60-character windows → next character** (slide 76).
3. Train an **Embedding → LSTM(256) → softmax** model (slide 77), watching samples improve during training.
4. Generate new text at different **temperatures** (slide 78).

*Runtime: Runtime ▸ Change runtime type ▸ **T4 GPU** required (20 epochs).*

---
## Step 1: Load the text and map each character to an integer (slide 75)

In [ ]:
import numpy as np
import keras
from keras import layers

keras.utils.set_random_seed(7)
path = keras.utils.get_file("shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")
text = open(path, encoding="utf-8").read()

chars = sorted(set(text))                      # 65 unique characters
char2idx = {c: i for i, c in enumerate(chars)}
ids = np.array([char2idx[c] for c in text])
print(len(text), "characters,", len(chars), "unique")
print(repr("".join(chars)))

## Step 2: Cut the text into 60-character windows (slide 76)

In [ ]:
seq_len, step = 60, 3                          # 60 chars -> next char
starts = np.arange(0, len(ids) - seq_len, step)
X = np.stack([ids[s:s + seq_len] for s in starts])
y = ids[starts + seq_len]
print(X.shape, y.shape)                        # (371778, 60) (371778,)

print("input :", repr("".join(chars[i] for i in X[0])))
print("target:", repr(chars[y[0]]))

## Step 3: Define and train the LSTM (slide 77)
The output layer has one neuron per character (65) with softmax.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(seq_len,)),
    layers.Embedding(len(chars), 32),
    layers.LSTM(256),
    layers.Dense(len(chars), activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.summary()

## Step 4: Generation functions (slide 78)
We define them **before** training so we can print a short sample every 5 epochs and watch the model learn to write.
The seed must be at least 60 characters long, because the model always reads the last 60 characters.

In [ ]:
def sample(probs, temperature=1.0):
    logits = np.log(probs + 1e-9) / temperature
    p = np.exp(logits - logits.max())
    return np.random.choice(len(p), p=p / p.sum())

def generate(seed, n_chars=300, temperature=0.5):
    out = seed
    for _ in range(n_chars):
        x = np.array([[char2idx[c] for c in out[-seq_len:]]])
        probs = model.predict(x, verbose=0)[0]
        out += chars[sample(probs, temperature)]
    return out

seed = ("ROMEO:\nBut, soft! what light through yonder window breaks?\n"
        "It is the east, and Juliet is the sun.\n")   # 98 chars >= 60
print(len(seed), "characters in the seed")

In [ ]:
# Print a short sample every 5 epochs
def show_sample(epoch, logs):
    if (epoch + 1) % 5 == 0:
        print(f"\n--- sample after epoch {epoch + 1} (temperature 0.5) ---")
        print(generate(seed, n_chars=150, temperature=0.5)[len(seed):])

history = model.fit(X, y, batch_size=256, epochs=20,     # use a GPU (e.g. Colab)
                    callbacks=[keras.callbacks.LambdaCallback(on_epoch_end=show_sample)])

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history["loss"], "b-o")
plt.xlabel("Epoch"); plt.ylabel("Training loss"); plt.title("Character-level LSTM"); plt.show()

## Step 5: Generate new text at three temperatures (slide 78)

In [ ]:
for t in (0.2, 0.5, 1.0):
    print(f"--- temperature {t} ---")
    print(generate(seed, temperature=t))

---
## Try it yourself
1. Which temperature produces the most Shakespeare-like text? What goes wrong at 0.2 and at 1.0?
2. Write your own seed of at least 60 characters, e.g. starting with `"KING HENRY:\n"`.
3. Replace `LSTM(256)` with `GRU(256)`. Is training faster? Is the text as good?
4. Save the trained model with `model.save("char_lstm.keras")` so you can generate text later without retraining.

In [ ]:
# Use this cell for your experiments